In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# import matplotlib.colors as mcolors
import pickle
import tikzplotlib
from collections import defaultdict
def tikzplotlib_fix_ncols(obj):
    """
    workaround for matplotlib 3.6 renamed legend's _ncol to _ncols, which breaks tikzplotlib
    """
    if hasattr(obj, "_ncols"):
        obj._ncol = obj._ncols
    for child in obj.get_children():
        tikzplotlib_fix_ncols(child)
casename="henon"

In [2]:
if casename=="logistic":
    from reservoirpy.datasets import logistic_map
    ts=logistic_map(100000, r=3.9, x0=0.5).flatten()
elif casename=="henon":
    from run_reservoir import henon1d   
    ts = henon1d(10000)

In [ ]:
num_qubits=6
num_meas=3
degree=2

In [ ]:
def get_data(ep, num_qubits, num_meas, method, noise="None", decode=True):
    degree=num_meas
    num_reservoirs=20
    timeplex=1
    string_identifier=str(ep)+"_casename"+str(casename)+"_num_qubits"+str(num_qubits)+"_num_meas"+str(num_meas)
    string_identifier+="_degree"+str(degree)+"_num_reservoirs"+str(num_reservoirs)+"_timeplex"+str(timeplex)
    string_identifier+="_method"+str(method)+"_noise"+str(noise)
    if not decode:
        string_identifier+="_decodeFalse"
    string_identifier+=".pickle"
    name="X_train"+string_identifier
    # print("name=",name)
    with open(name,"rb") as f:
        X_train = pickle.load(f)
    
    name="X_test"+string_identifier
    with open(name,"rb") as f:
        X_test = pickle.load(f)
    
    name="score"+string_identifier
    with open(name,"rb") as f:
        score = pickle.load(f)
        
    name="prediction"+string_identifier
    with open(name,"rb") as f:
        prediction = pickle.load(f)
        
    name="state"+string_identifier
    with open(name,"rb") as f:
        state = pickle.load(f)
    return X_train, X_test, score, prediction, state, string_identifier

In [4]:
# X_train, X_test, score, prediction, state, basen = get_data(ep, 7, 6, "classical", "None")
# _=plt.plot(state)

In [ ]:
for ep in [0,4]:
    X_train, X_test, score, prediction, states, basen = get_data(ep, num_qubits, num_meas, "classical", "None")
    data=states
    groups = defaultdict(list)
    for n, m, _ , stats in data:
        mean, var = stats
        x = n
        groups[n].append((x, float(mean), float(var)))

    # Plot each group
    plt.figure(figsize=(8,6))

    for n, values in groups.items():
        values.sort(key=lambda v: v[0])  # sort by x = n/m
        xs = [v[0] for v in values]
        ys = [v[1] for v in values]
        errs = [v[2] for v in values]
        plt.errorbar(xs, ys, yerr=errs, label=f"n = {n}", marker='o', capsize=4)

    plt.xlabel("degree")
    plt.ylabel("Mean ± Variance")
    plt.legend()
    plt.grid(True)
    plt.show()